# Household Object AI — train YOLO26n on a free Colab GPU

This notebook fine-tunes the pretrained **YOLO26n** model on the project's **15-class household dataset**
(`data/household/`) using a free **NVIDIA T4 GPU**, then validates the result and lets you download `best.pt`.

**Run the cells in order, top to bottom** (click a cell, then press **Shift + Enter**).

| | |
|---|---|
| Dataset | 5,080 train / 906 val images, 15 classes, ~15% val per class |
| Config (from `docs/model-training.md`) | `epochs=100  imgsz=640  batch=16  patience=20` |
| Expected time on a T4 | roughly **1–3 hours** (early stopping may finish sooner) |
| Output | saved to your Google Drive, so it survives a disconnect |

> ⚠️ **Pressure cooker (class 12) has 0 training and 0 validation images.**
> It stays in the class list on purpose (class IDs must not change), but the model **cannot learn it in
> this run**. Validation will report no results for it. It will be trained later with custom photos.

This notebook does **not** change `data.yaml`, class IDs or labels, and does **not** deploy anything.

## Before you start (on your laptop) — what to upload

You do **not** need the Git repository in Colab — only one zip file (≈ 840 MB).

1. On the laptop, create the zip (skip if `data/_cache/colab/household_dataset.zip` already exists):
   ```powershell
   cd D:\household-object-ai
   backend\.venv\Scripts\python.exe tools\dataset\07_package_for_colab.py
   ```
2. Open **Google Drive** in the browser and create a folder **`household-object-ai`** in *My Drive*.
3. Upload into that folder:

   | Local file | Upload to Google Drive | Required? |
   |---|---|---|
   | `D:\household-object-ai\data\_cache\colab\household_dataset.zip` | `My Drive/household-object-ai/household_dataset.zip` | **Yes** |
   | `D:\household-object-ai\data\_cache\colab\household_dataset.manifest.json` | `My Drive/household-object-ai/household_dataset.manifest.json` | Recommended (verifies the upload) |
   | `D:\household-object-ai\backend\models\yolo26n.pt` | `My Drive/household-object-ai/yolo26n.pt` | Optional (otherwise the official file is downloaded and checked) |

4. In Colab: **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**.

When training finishes, the trained model is in
**`My Drive/household-object-ai/runs/household-yolo26n/weights/best.pt`** (and the last cell also downloads it).

## Step 1 — Check that a GPU is available

In [ ]:
!nvidia-smi

import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU found. In the menu choose Runtime → Change runtime type → T4 GPU, then run this cell again.")
DEVICE = 0
GPU_NAME = torch.cuda.get_device_name(0)
print(f"✅ CUDA available — GPU: {GPU_NAME} | torch {torch.__version__} | CUDA {torch.version.cuda}")

## Step 2 — Install Ultralytics
Colab already has PyTorch with CUDA. This installs the same Ultralytics version the project uses locally (8.4.166).

In [ ]:
%pip install -q "ultralytics==8.4.166"

import ultralytics
ultralytics.checks()

## Step 3 — Connect Google Drive and set paths
A pop-up asks for permission to access your Drive — allow it.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/household-object-ai")   # the folder you created
ZIP_ON_DRIVE = DRIVE_DIR / "household_dataset.zip"
MANIFEST_ON_DRIVE = DRIVE_DIR / "household_dataset.manifest.json"
WEIGHTS_ON_DRIVE = DRIVE_DIR / "yolo26n.pt"                       # optional upload
RUNS_DIR = DRIVE_DIR / "runs"                                     # training output (kept on Drive)
RUN_NAME = "household-yolo26n"

LOCAL_ROOT = Path("/content/work")          # fast local disk for the dataset
DATA_ROOT = LOCAL_ROOT / "household"
DATA_YAML = DATA_ROOT / "data.yaml"

if not ZIP_ON_DRIVE.exists():
    raise SystemExit(f"Not found: {ZIP_ON_DRIVE}\nUpload household_dataset.zip to My Drive/household-object-ai/ and re-run.")
RUNS_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
print(f"✅ Found dataset zip ({ZIP_ON_DRIVE.stat().st_size / 1e6:.0f} MB)")
print("Training output will be saved to:", RUNS_DIR / RUN_NAME)

## Step 4 — Copy, verify and unzip the dataset
Copies the zip to Colab's local disk (training reads much faster from there), checks it, and unzips.

In [ ]:
import hashlib, json, shutil, zipfile
from collections import Counter

local_zip = LOCAL_ROOT / "household_dataset.zip"
if not local_zip.exists() or local_zip.stat().st_size != ZIP_ON_DRIVE.stat().st_size:
    print("Copying zip from Drive (1–3 minutes)...")
    shutil.copy(ZIP_ON_DRIVE, local_zip)

manifest = json.loads(MANIFEST_ON_DRIVE.read_text()) if MANIFEST_ON_DRIVE.exists() else None
if manifest:
    h = hashlib.sha256()
    with open(local_zip, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    if h.hexdigest() != manifest["sha256"]:
        raise SystemExit("❌ The zip is corrupted or different from the laptop's copy. Upload it again.")
    print("✅ Zip checksum matches the laptop's manifest")
else:
    print("ℹ️ No manifest uploaded — skipping the checksum check")

if not DATA_YAML.exists():
    print("Unzipping...")
    with zipfile.ZipFile(local_zip) as zf:
        zf.extractall(LOCAL_ROOT)

# ---- verify structure and counts (read-only: nothing in the dataset is changed)
import yaml
cfg = yaml.safe_load(DATA_YAML.read_text())
NAMES = [cfg["names"][i] for i in sorted(cfg["names"])]
assert cfg["nc"] == 15 and len(NAMES) == 15, "data.yaml must have 15 classes"
assert NAMES[12] == "pressure cooker", "class 12 must be 'pressure cooker'"

expected = manifest["splits"] if manifest else {}
per_class = {}
for split in ("train", "val"):
    imgs = list((DATA_ROOT / "images" / split).glob("*.jpg"))
    labs = list((DATA_ROOT / "labels" / split).glob("*.txt"))
    assert len(imgs) == len(labs), f"{split}: images and labels don't match"
    if expected:
        assert len(imgs) == expected[split]["images"], f"{split}: expected {expected[split]['images']} images, found {len(imgs)}"
    counts = Counter()
    for lab in labs:
        counts.update({int(line.split()[0]) for line in lab.read_text().splitlines() if line.strip()})
    per_class[split] = counts
    print(f"✅ {split}: {len(imgs):,} images")

print(f"\n{'id':>3}  {'class':16}{'train imgs':>11}{'val imgs':>10}")
for i, name in enumerate(NAMES):
    note = "   ⚠️ NO DATA — cannot be learned in this run" if per_class["train"][i] == 0 else ""
    print(f"{i:>3}  {name:16}{per_class['train'][i]:>11}{per_class['val'][i]:>10}{note}")

from ultralytics.data.utils import check_det_dataset
check_det_dataset(str(DATA_YAML))   # Ultralytics' own validation of data.yaml + paths
print("\n✅ Dataset ready:", DATA_YAML)

## Step 5 — Get the pretrained YOLO26n weights
Uses your uploaded `yolo26n.pt` if present; otherwise downloads the official file.
Either way it is checked against the laptop's copy.

In [ ]:
import hashlib, shutil
from ultralytics import YOLO

EXPECTED_SHA256 = "9b09cc8bf347f0fc8a5f7657480587f25db09b34bf33b0652110fb03a8ad4fef"  # backend/models/yolo26n.pt
WEIGHTS = LOCAL_ROOT / "yolo26n.pt"

if WEIGHTS_ON_DRIVE.exists():
    shutil.copy(WEIGHTS_ON_DRIVE, WEIGHTS)
    print("Using yolo26n.pt uploaded to Drive")
elif not WEIGHTS.exists():
    import os
    os.chdir(LOCAL_ROOT)
    YOLO("yolo26n.pt")                     # downloads the official weights into /content/work
    print("Downloaded official yolo26n.pt")

digest = hashlib.sha256(WEIGHTS.read_bytes()).hexdigest()
if digest == EXPECTED_SHA256:
    print("✅ Weights match the project's backend/models/yolo26n.pt")
else:
    print("⚠️ Weights differ from the laptop's file (Ultralytics may have updated them).\n"
          "   For an exact match, upload backend/models/yolo26n.pt to My Drive/household-object-ai/ and re-run this cell.")
print("Weights:", WEIGHTS)

## Step 6 — Train  ⏱️ (1–3 hours)
Uses exactly the configuration from `docs/model-training.md`. Keep this browser tab open.
Progress is saved to Drive after every epoch; if Colab disconnects, use **Step 6b**.

`project` is an absolute Drive path so the run lands exactly in `runs/household-yolo26n`
(with a relative path, Ultralytics 8.4 would nest it under `runs/detect/`).

In [ ]:
from ultralytics import YOLO

if (RUNS_DIR / RUN_NAME / "weights" / "last.pt").exists():
    raise SystemExit(f"A run already exists at {RUNS_DIR / RUN_NAME}.\n"
                     "To continue it, use Step 6b. To start fresh, rename or delete that folder on Drive first.")

model = YOLO(str(WEIGHTS))
results = model.train(
    data=str(DATA_YAML),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    device=DEVICE,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,      # safe: the check above stops if a real checkpoint already exists
    workers=2,          # Colab gives 2 CPU cores
)
TRAIN_DIR = Path(model.trainer.save_dir)
print("✅ Training finished. Run folder:", TRAIN_DIR)

## Step 6b — ONLY if Colab disconnected during training
Re-run Steps 1–5 first, then set `RESUME = True` below and run this cell. Skip it otherwise.

In [ ]:
RESUME = False   # change to True only to continue an interrupted run

if RESUME:
    last = RUNS_DIR / RUN_NAME / "weights" / "last.pt"
    if not last.exists():
        raise SystemExit(f"No checkpoint at {last} — start with Step 6 instead.")
    model = YOLO(str(last))
    model.train(resume=True)
    TRAIN_DIR = Path(model.trainer.save_dir)
    print("✅ Resumed training finished. Run folder:", TRAIN_DIR)
else:
    print("Skipped (RESUME is False).")

## Step 7 — Check that training completed and find `best.pt`

In [ ]:
import csv

TRAIN_DIR = Path(globals().get("TRAIN_DIR", RUNS_DIR / RUN_NAME))   # set by Step 6/6b in this session
BEST_PT = TRAIN_DIR / "weights" / "best.pt"
if not BEST_PT.exists():
    raise SystemExit(f"❌ best.pt not found at {BEST_PT}. Training did not finish — see Step 6/6b.")

rows = list(csv.DictReader(open(TRAIN_DIR / "results.csv")))
epochs_done = len(rows)
best = max(rows, key=lambda r: float(r["metrics/mAP50-95(B)"].strip() or 0))
stopped_early = epochs_done < 100
print(f"GPU:                 {GPU_NAME}")
print(f"Training status:     {'✅ completed' + (' (early stopping)' if stopped_early else '')} — {epochs_done} epochs")
print(f"Best epoch:          {best['epoch'].strip()}  (val mAP50-95 {float(best['metrics/mAP50-95(B)']):.3f})")
print(f"best.pt:             {BEST_PT}  ({BEST_PT.stat().st_size / 1e6:.1f} MB)")

## Step 8 — Validate `best.pt` on the validation set

In [ ]:
best_model = YOLO(str(BEST_PT))
metrics = best_model.val(
    data=str(DATA_YAML),
    imgsz=640,
    batch=16,
    device=DEVICE,
    split="val",
    project=str(RUNS_DIR),
    name=f"{RUN_NAME}-val",
    exist_ok=True,
    plots=True,
)

print("\n================ VALIDATION RESULTS ================")
print(f"GPU:          {GPU_NAME}")
print(f"best.pt:      {BEST_PT}")
print(f"mAP50:        {metrics.box.map50:.4f}")
print(f"mAP50-95:     {metrics.box.map:.4f}")
print(f"Precision:    {metrics.box.mp:.4f}  (mean over classes with val data)")
print(f"Recall:       {metrics.box.mr:.4f}")

rows = {r["Class"]: r for r in metrics.summary()}
print(f"\n{'id':>3}  {'class':16}{'images':>7}{'boxes':>7}{'P':>8}{'R':>8}{'mAP50':>8}{'mAP50-95':>10}")
weak = []
for i, name in enumerate(NAMES):
    r = rows.get(name)
    if r is None:
        print(f"{i:>3}  {name:16}{'—':>7}{'—':>7}   no validation data (not learnable in this run)")
        continue
    print(f"{i:>3}  {name:16}{r['Images']:>7}{r['Instances']:>7}{r['Box-P']:>8.3f}{r['Box-R']:>8.3f}"
          f"{r['mAP50']:>8.3f}{r['mAP50-95']:>10.3f}")
    if r["mAP50"] < 0.5:
        weak.append(name)
if weak:
    print("\n⚠️ Classes with mAP50 < 0.50 (need more / better data):", ", ".join(weak))
print("\nPlots (confusion matrix, PR curves):", RUNS_DIR / f"{RUN_NAME}-val")

## Step 9 — Download `best.pt` (and the training report)
Saves a clearly named copy on Drive and downloads a zip to your computer.

In [ ]:
import shutil
from google.colab import files

named_copy = DRIVE_DIR / "household-yolo26n-best.pt"
shutil.copy(BEST_PT, named_copy)
print("✅ Copy on Drive:", named_copy)

bundle_dir = LOCAL_ROOT / "household-yolo26n-results"
bundle_dir.mkdir(exist_ok=True)
for name in ("results.csv", "results.png", "args.yaml", "confusion_matrix.png", "confusion_matrix_normalized.png"):
    for src in (TRAIN_DIR / name, RUNS_DIR / f"{RUN_NAME}-val" / name):
        if src.exists():
            shutil.copy(src, bundle_dir / (src.parent.name + "__" + name))
shutil.copy(BEST_PT, bundle_dir / "best.pt")
archive = shutil.make_archive(str(LOCAL_ROOT / "household-yolo26n-results"), "zip", bundle_dir)
files.download(archive)      # your browser downloads household-yolo26n-results.zip
print("⬇️ Downloading:", archive)

## Back on the laptop — where to put the trained model

1. Unzip `household-yolo26n-results.zip` (from your browser's Downloads folder), or take
   `My Drive/household-object-ai/household-yolo26n-best.pt`.
2. Put the model at the path the project docs use (create the folders if needed):
   ```
   D:\household-object-ai\backend\runs\household-yolo26n\weights\best.pt
   ```
   (`backend/runs/` is git-ignored, so the weights will not be committed.)
3. **Do not deploy yet:** leave `backend/models/yolo26n.pt` and `backend/.env` (`MODEL_PATH`) unchanged.
   The next step is comparing this model with the current YOLO26n on real household photos.

**Reminder:** pressure cooker (class 12) had no data, so this model will not detect it.